## Dyscalculia diagnosis-report notes

Summarizes Maike's recruitment notes from the dyscalculics' diagnosis reports (`Abklärungsbericht`), restricted to the subjects who ended up in the final scanned sample (`group == 1`).

Source table: `add_tables/ dyscalc_recruit-overview_Abklärungsbericht-notes.csv` (note the leading space in the filename).
It has two junk header rows, then the real header on row 4, with the columns of interest being:
- `gI` — free-text notes on additional diagnoses / remarks (transcribed from the report's `Bemerkungen` section)
- `tests` — free-text notes on where the subject scored below/average/above average across the cognitive tests in the report

Subjects are matched to the final sample by first+last name, since that's the only identifier both tables share.

In [1]:
import pandas as pd
import numpy as np
import os.path as op
import re

pati = '/Users/mrenke/data/ds-dnumrisk/add_tables'
target_folder = '/Users/mrenke/data/ds-dnumrisk/derivatives/phenotype'

/Users/mrenke/mambaforge/envs/behav_fit/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### Load & match the two source tables

In [2]:
notes = pd.read_csv(op.join(pati, ' dyscalc_recruit-overview_Abklärungsbericht-notes.csv'), skiprows=3, header=0)
notes = notes[['Vorname', 'Familienname', 'Kispi ID', 'gI', 'tests']].dropna(subset=['Vorname'])
notes['full_name_norm'] = (notes['Vorname'].astype(str).str.strip() + ' ' + notes['Familienname'].astype(str).str.strip()).str.lower()
notes.head()

,Vorname,Familienname,Kispi ID,gI,tests,full_name_norm
0,Carlota,Vega,96 / 101,"""gute Schülerin""",duch. - überdurch; except VS,carlota vega
1,Neela,Haenni,94 / 99,NaN,durch.; except VS-erfassunsspanne; überduch in...,neela haenni
2,Alvin,Krasniqi,93 / 98,NaN,durch.,alvin krasniqi
3,Jessica,Mächler,91 / 96,NaN,durch. ; except exekutive Funktionen,jessica mächler
4,Thalia,Arn Ilenia,87 / 91,NaN,NaN,thalia arn ilenia


In [3]:
final = pd.read_csv(op.join(pati, 'subjects_recruit&scan_scanned_final.csv'))
final.columns = final.iloc[0]
final = final[1:].reset_index(drop=True)
final['name'] = final['name'].str.strip()

dys = final[final['group'] == '1'].copy()
dys = dys[['subject ID', 'name']].rename(columns={'subject ID': 'subject'}).astype({'subject': int})
print(f'{len(dys)} dyscalculic subjects in the final sample')
dys.head()

33 dyscalculic subjects in the final sample


,subject,name
1,2,Sophia Atzori
3,4,Carlota Vega
5,6,Neela Haenni
7,8,Jeanne Hoeltschi
10,11,Chi Tanner


The names weren't always transcribed identically between the two tables (nicknames, middle names, spelling variants, or a subject who changed their name/gender marker between recruitment and scanning). Close matches were checked by hand against the diagnosis-notes table and resolved in `manual_name_map` below — verify this if new subjects are added later.

In [4]:
manual_name_map = {
    'sophia atzori': 'sophia medea atzori',
    'jeanne hoeltschi': 'jeanne höltschi',
    'chi tanner': 'medea (will chi/er genannt werden) tanner',   # subject wished to be called Chi
    'silas jörg': 'shayenne kira jörg',                          # subject went by Silas at time of scanning
    'tamara tremonte': 'tamara tremont',
    'nina flachsman': 'nina flachsmann',
    'kaj hausdorff': 'kaj annika hausdorff',
    'shan-shan leys': 'shan leys',
    'ella senti': 'ella senti (sommer)',
    'laura jäger': 'laura sophia jäger',
}

def match_name(name):
    key = name.lower()
    key = manual_name_map.get(key, key)
    match = notes[notes['full_name_norm'] == key]
    return match.iloc[0] if len(match) else None

matched_rows = []
unmatched = []
for _, row in dys.iterrows():
    m = match_name(row['name'])
    if m is None:
        unmatched.append(row['name'])
        continue
    matched_rows.append({
        'subject': row['subject'],
        'name': row['name'],  # kept only for QC in this notebook, dropped before saving
        'kispi_id': m['Kispi ID'],
        'other_diagnoses_notes': m['gI'],
        'test_performance_raw': m['tests'],
    })

assert not unmatched, f'unmatched names, check manual_name_map: {unmatched}'
df = pd.DataFrame(matched_rows)
df.head()

,subject,name,kispi_id,other_diagnoses_notes,test_performance_raw
0,2,Sophia Atzori,76 / 80,"Hinweise auf Schwankungen der Aufmerksamkeit, ...",unterdurch. Inhibition
1,4,Carlota Vega,96 / 101,"""gute Schülerin""",duch. - überdurch; except VS
2,6,Neela Haenni,94 / 99,NaN,durch.; except VS-erfassunsspanne; überduch in...
3,8,Jeanne Hoeltschi,57/ 60,NaN,druch
4,11,Chi Tanner,59 / 62,NaN,"unter - matrizen, vis. puzzle; über in L&S"


### Parse the `tests` free text into a level + exceptions column

Maike's notes use `unter(durch)` / `durch` / `über(durch)` (short for *unterdurchschnittlich* / *durchschnittlich* / *überdurchschnittlich*, i.e. below-/average/above-average) as the overall marker, sometimes followed by which specific (sub)test(s) deviated from that (e.g. `except VS`, `unter - matrizen, vis. puzzle`).

This is **not** a strictly structured format (typos like `duch`/`druch`, inconsistent use of `;` vs `,`, mixed levels in one note), so the parsing below is a best-effort heuristic:
- `test_overall_level`: which of below-/average/above-average (or a combination, if the note itself says performance was mixed) is mentioned
- `test_exceptions_detail`: the part of the note that names a specific (sub)test or uses a qualifier like `except`/`nur`, i.e. everything beyond a bare overall-level statement

`test_performance_raw` is always kept alongside so the parsed columns can be checked against the original wording.

In [5]:
BELOW_RE = re.compile(r'unterdurch\w*|unterduch\w*|\bunter\b\.?')
ABOVE_RE = re.compile(r'überdurch\w*|überduch\w*|\büber\b\.?')
AVERAGE_RE = re.compile(r'\b(durch|duch|druch)\b\.?\w*')
TRIGGER_WORDS = ['except', 'nur', 'außer', 'ausser']

def classify_tests(text):
    if pd.isna(text) or not str(text).strip():
        return pd.Series({'test_overall_level': None, 'test_exceptions_detail': None})

    clauses = [c.strip() for c in re.split(r';', str(text)) if c.strip()]
    levels_found = []
    exception_clauses = []

    for clause in clauses:
        low = clause.lower()
        has_below = bool(BELOW_RE.search(low))
        has_above = bool(ABOVE_RE.search(low))
        low_no_extremes = ABOVE_RE.sub(' ', BELOW_RE.sub(' ', low))
        has_average = bool(AVERAGE_RE.search(low_no_extremes))

        if has_below:
            levels_found.append('below_average')
        if has_above:
            levels_found.append('above_average')
        if has_average:
            levels_found.append('average')

        # a clause is 'exception detail' if it flags an exception explicitly, or
        # still contains substantive text (a named test, digits, etc.) once the
        # bare level words are stripped out
        remainder = AVERAGE_RE.sub(' ', low_no_extremes)
        remainder = re.sub(r'[^a-zäöüß0-9&\-\s]', ' ', remainder)
        remainder_words = [w for w in remainder.split() if len(w) > 2 or w.isdigit()]
        is_trigger = any(t in low for t in TRIGGER_WORDS) or '(' in clause

        if is_trigger or remainder_words:
            exception_clauses.append(clause)

    unique_levels = sorted(set(levels_found), key=levels_found.index)
    if not unique_levels:
        overall = 'not_specified'
    elif len(unique_levels) == 1:
        overall = unique_levels[0]
    else:
        overall = 'mixed (' + '/'.join(unique_levels) + ')'

    exceptions = '; '.join(exception_clauses) if exception_clauses else None
    return pd.Series({'test_overall_level': overall, 'test_exceptions_detail': exceptions})


df[['test_overall_level', 'test_exceptions_detail']] = df['test_performance_raw'].apply(classify_tests)
df[['name', 'test_performance_raw', 'test_overall_level', 'test_exceptions_detail']]

,name,test_performance_raw,test_overall_level,test_exceptions_detail
0,Sophia Atzori,unterdurch. Inhibition,below_average,unterdurch. Inhibition
1,Carlota Vega,duch. - überdurch; except VS,mixed (above_average/average),except VS
2,Neela Haenni,durch.; except VS-erfassunsspanne; überduch in...,mixed (average/above_average),except VS-erfassunsspanne; überduch in visuomo...
3,Jeanne Hoeltschi,druch,average,None
4,Chi Tanner,"unter - matrizen, vis. puzzle; über in L&S",mixed (below_average/above_average),"unter - matrizen, vis. puzzle; über in L&S"
5,Nina Bürge,NaN,None,None
6,Silas Jörg,"durch. (paar unter & über, without clear pattern)",mixed (below_average/above_average/average),"durch. (paar unter & über, without clear pattern)"
7,Alma Nikokochev,druch.,average,None
8,Iva Kuhac,"unter - matrizen, vis. puzzle; über in Kategor...",mixed (below_average/above_average),"unter - matrizen, vis. puzzle; über in Kategor..."
9,Michelle Rübel,unter NUR in Basis-Math(mathemathik),below_average,unter NUR in Basis-Math(mathemathik)


Sanity check: how many subjects fall into each overall level, and how many have no test notes at all (recruited late, report notes never filled in).

In [6]:
df['test_overall_level'].fillna('no notes available').value_counts()

test_overall_level
no notes available                             11
average                                         6
below_average                                   5
mixed (below_average/above_average/average)     5
mixed (below_average/above_average)             2
mixed (above_average/average)                   1
mixed (average/above_average)                   1
not_specified                                   1
above_average                                   1
Name: count, dtype: int64

### Save

Dropping `name` before saving to keep this consistent with the other (anonymized, subject-ID-only) tables in `derivatives/phenotype/`. The name -> notes matching above is fully reproducible from the two source tables, so it can always be re-derived/checked here if needed.

In [7]:
df_out = df.drop(columns=['name']).sort_values('subject').reset_index(drop=True)
df_out.to_csv(op.join(target_folder, 'dyscalc_diagnosis-notes_summary.csv'), index=False)
df_out

,subject,kispi_id,other_diagnoses_notes,test_performance_raw,test_overall_level,test_exceptions_detail
0,2,76 / 80,"Hinweise auf Schwankungen der Aufmerksamkeit, ...",unterdurch. Inhibition,below_average,unterdurch. Inhibition
1,4,96 / 101,"""gute Schülerin""",duch. - überdurch; except VS,mixed (above_average/average),except VS
2,6,94 / 99,NaN,durch.; except VS-erfassunsspanne; überduch in...,mixed (average/above_average),except VS-erfassunsspanne; überduch in visuomo...
3,8,57/ 60,NaN,druch,average,None
4,11,59 / 62,NaN,"unter - matrizen, vis. puzzle; über in L&S",mixed (below_average/above_average),"unter - matrizen, vis. puzzle; über in L&S"
5,13,23/25,NaN,NaN,None,None
6,16,49/52,NaN,"durch. (paar unter & über, without clear pattern)",mixed (below_average/above_average/average),"durch. (paar unter & über, without clear pattern)"
7,17,27/29,Hinweis auf Schweirigkeiten in geteilter Aufme...,druch.,average,None
8,22,72 / 75,"Hinweis auf visuell-räumliche Probleme, Psychi...","unter - matrizen, vis. puzzle; über in Kategor...",mixed (below_average/above_average),"unter - matrizen, vis. puzzle; über in Kategor..."
9,23,13/15,sehr gute Schülerin,unter NUR in Basis-Math(mathemathik),below_average,unter NUR in Basis-Math(mathemathik)
